# Phase 2.2 — Fine-tune HingBERT (single seed) — validation run

**Goal:** confirm the SA pipeline is correct end-to-end and that HingBERT lands in the
**~63–66 macro-F1** sanity band *before* scaling to 5 models x 3 seeds.

**Prereq:** Phase 2.1 has written `data/sa_master.parquet` and `data/label_mapping.json`
to Drive. Columns expected in the master: `sentence_id, label, tokens, lang_tags, text,
label_id, split, cmi`. Label map: `{negative:0, neutral:1, positive:2}`.

**Design notes**
- HingBERT is public (no `HF_TOKEN` needed — that's only for gated IndicBERT).
- `pred_label`/`true_label` are written as **integer ids**; decode with `label_mapping.json` in Phase 4.
- `sentence_id` is carried through and never mutated; row counts are asserted after prediction.
- Uses `eval_strategy=` (transformers >= 4.41), matching the version floor in the plan.


In [ ]:
!pip -q install "transformers>=4.41" datasets accelerate scikit-learn pandas pyarrow sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, numpy as np, pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

PROJECT = "/content/drive/MyDrive/codemix_project"
DATA    = f"{PROJECT}/data"
CKPT    = f"{PROJECT}/checkpoints"
PREDS   = f"{PROJECT}/preds"
for d in (CKPT, PREDS):
    os.makedirs(d, exist_ok=True)

MODEL_ID   = "l3cube-pune/hing-bert"
MODEL_NAME = "hing-bert"      # used in checkpoint dir + pred CSV
SEED       = 42
MAX_LEN    = 128              # tweets are short; 128 tokens is ample

set_seed(SEED)
print("CUDA:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# --- load master + label mapping, split, and guard the schema -----------------
df = pd.read_parquet(f"{DATA}/sa_master.parquet")
with open(f"{DATA}/label_mapping.json") as f:
    label2id = json.load(f)                 # {"negative":0,"neutral":1,"positive":2}
id2label = {int(v): k for k, v in label2id.items()}

print("rows:", len(df), "| columns:", list(df.columns))
print(df["split"].value_counts())

assert {"sentence_id", "text", "label_id", "split"}.issubset(df.columns), "master schema drift"
assert set(df["label_id"].unique()).issubset(set(id2label)), "label_id not in mapping"
assert df["sentence_id"].is_unique, "sentence_id must be unique (immutable-key discipline)"

train_df = df[df.split == "train"].reset_index(drop=True)

val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

In [ ]:
# --- tokenize; only scalar columns go into the HF dataset ---------------------
# (keeping tokens/lang_tags/cmi out avoids Arrow list-column friction; we recover
#  sentence_id + true labels from *_df by row order, which map() and predict() preserve)
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def to_ds(frame):
    ds = Dataset.from_pandas(frame[["sentence_id", "text", "label_id"]], preserve_index=False)
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    ds = ds.rename_column("label_id", "labels")
    return ds

train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)
collator = DataCollatorWithPadding(tok)   # dynamic padding per batch

In [ ]:
# --- model, metrics, trainer --------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=3, id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir=f"{CKPT}/{MODEL_NAME}_seed{SEED}",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",           # requires transformers >= 4.41
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tok, data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
# --- evaluate + sanity-band check --------------------------------------------
val_metrics = trainer.evaluate(val_ds)
print("VAL:", {k: round(v, 4) for k, v in val_metrics.items() if k.startswith("eval_")})

test_out   = trainer.predict(test_ds)                 # order matches test_df
test_preds = np.argmax(test_out.predictions, axis=-1)
test_acc   = accuracy_score(test_df["label_id"], test_preds)
test_f1    = f1_score(test_df["label_id"], test_preds, average="macro")
print(f"TEST  acc={test_acc:.4f}  macro-F1={test_f1:.4f}")

if 0.60 <= test_f1 <= 0.68:
    print("\u2713 In/near the 63-66 band \u2014 pipeline looks correct. Proceed to 5x3 scaling.")
elif test_f1 < 0.45:
    print("\u2717 Near-chance F1 \u2014 debug label mapping / tokenizer BEFORE scaling.")
else:
    print("~ Outside band but not collapsed \u2014 inspect per-class F1 below.")

print(classification_report(test_df["label_id"], test_preds,
      target_names=[id2label[i] for i in range(3)], digits=4))

In [ ]:
# --- per-sentence predictions (the load-bearing Phase 2 output) --------------
preds_df = pd.DataFrame({
    "sentence_id": test_df["sentence_id"].values,   # unchanged from the master
    "model":       MODEL_NAME,
    "seed":        SEED,
    "true_label":  test_df["label_id"].values,      # int ids; decode via label_mapping.json in Phase 4
    "pred_label":  test_preds,
})
preds_df["correct"] = (preds_df.true_label == preds_df.pred_label).astype(int)

assert len(preds_df) == len(test_df), "row count changed -> key discipline violated"
assert preds_df["sentence_id"].is_unique, "duplicate sentence_id in predictions"

out = f"{PREDS}/{MODEL_NAME}_seed{SEED}.csv"
preds_df.to_csv(out, index=False)
print("wrote", out, preds_df.shape)
preds_df.head()

## What to check before scaling to 2.3–2.4

1. **F1 in the ~63–66 band** on TEST. If it collapses to ~33 (chance for 3-class), the
   fault is almost always the label mapping or the tokenizer — fix that first.
2. **Per-class report** isn't degenerate (no class at 0.00 F1).
3. `preds/hing-bert_seed42.csv` exists with the exact columns
   `sentence_id, model, seed, true_label, pred_label, correct` and the same row count as the test split.
4. Checkpoints landed under `checkpoints/hing-bert_seed42/` on Drive (survives disconnect).

Once this passes, 2.3–2.4 is this same body wrapped in a loop over the 5 model ids and
seeds `[42, 43, 44]`, writing one `preds/{model}_seed{seed}.csv` each. IndicBERT is the
only one needing `HF_TOKEN` + accepted terms, and is expected to score lower by design.
